# RL-based model combiner for live IDS verdicts

This project has trained 10 different models scored side-by-side on live
traffic (deployed hybrid, experimental variants 1-3, the 3
`classifier_comparison` candidates, and the 3 attack-family specialists) -
but there's still no combination mechanism: each is scored independently,
and nothing decides a single final verdict from all 10 votes. That's the
"no combination mechanism defined" gap flagged as an open weakness of the
attack-family split.

This notebook trains a reinforcement-learning policy to do that
combination - framed as a **one-step contextual bandit** (each flow's
combination decision is independent of every other flow's, so a deeper
multi-step MDP would solve a problem that doesn't exist here).

## Trains on REAL data, not synthetic

Every `simulate_attacks.py` live-test run already records, per flow, each
available model's binary prediction and the scenario's ground truth. That
means - unlike a retrain/promotion decision, where this project has only
~10 real historical episodes on record - there's actually enough real
labeled data here to train directly on, no synthetic environment needed.

## Read this before pointing it at your own result directories

A live-test directory's per-flow `raw_flood`/`reflection`/etc. keys are
only comparable to another directory's if **both came from the same
trained checkpoint**. This project's own history has at least 4 distinct
`raw_flood`/`reflection` checkpoints reusing identical key names across
its debugging journey (benign-starved -> overcorrected -> final
ratio-balanced - see `CHANGELOG.md`). Checked directly by comparing each
candidate directory's per-scenario recall/specificity profile:

| Directory | raw_flood profile |
|---|---|
| `family_prefix_5trial` | 100% recall / 0% specificity - the benign-starved, flags-everything checkpoint |
| `family_fixed_5trial` | 0% recall / 100% specificity - the overcorrected, flags-nothing checkpoint |
| `other_models_5trial` | a third, distinct profile |
| `family_aware_5trial`, `final_balanced_5trial` | ~13-20% recall / ~75-98% specificity - the final checkpoint, consistent with each other |

Blindly pooling all of them (an earlier pass at this script did exactly
that) teaches the combiner that the same vote pattern means opposite
things - it never converges to anything coherent. `RESULT_DIRS` below
only pools the two directories confirmed to share the final checkpoint.
**Before adding your own directories to that list, run the same
per-scenario spot check** (a cell near the bottom does this for you) -
don't assume shared key names mean shared models.

## State / action / reward

- **State**: the 10 models' own binary predictions for one flow (1 =
  that model flagged ATTACK, 0 = NORMAL, 0.5 if unavailable for that flow
  - matches how the dashboard itself degrades when a variant fails to
  load).
- **Action**: 0 = NORMAL, 1 = ATTACK (the combiner's own final verdict).
- **Reward**: +1 correct / -1 incorrect (symmetric) by default; a second,
  asymmetric run (missing an attack costs more than a false alarm) is
  included since that's the operationally realistic version for an IDS -
  a cost ratio a plain accuracy-trained classifier can't express
  directly, but reward shaping can.

In [ ]:
import glob
import json
import os

import numpy as np

RNG = np.random.default_rng(42)

import numpy as np

RNG = np.random.default_rng(42)


## Config: which models, which result directories

See the warning above before editing `RESULT_DIRS`.

In [ ]:
MODEL_KEYS = [
    "deployed_hybrid", "variant1_xgb_single_flow", "variant2_xgb_temporal", "variant3_cnn_lstm",
    "xgboost", "random_forest", "histgradientboosting",
    "raw_flood", "reflection", "connection_application_layer",
]

# IMPORTANT - read before pointing this at your own result directories:
# a live-test directory's per-flow "raw_flood"/"reflection"/etc. keys are
# only comparable to another directory's if both were generated by the
# SAME trained checkpoint. This project's own history has at least 4
# distinct raw_flood/reflection checkpoints reusing identical key names
# across its debugging journey (benign-starved -> overcorrected -> final
# ratio-balanced, see CHANGELOG.md) - checked directly by comparing each
# candidate directory's per-scenario recall/specificity profile:
#   family_prefix_5trial   -> 100% recall / 0% specificity   (benign-starved)
#   family_fixed_5trial    -> 0% recall / 100% specificity   (overcorrected)
#   other_models_5trial    -> a third, distinct profile
#   family_aware_5trial,
#   final_balanced_5trial  -> ~13-20% recall / ~75-98% specificity (final, consistent with each other)
# Blindly pooling all of them (an earlier version of this script did)
# teaches the combiner that the same vote pattern means opposite things -
# it never converges. Only pool directories you've confirmed came from
# the SAME model checkpoint (e.g. by this same per-scenario spot check),
# not just ones that happen to share key names.
RESULT_DIRS = [
    "family_aware_5trial", "final_balanced_5trial",
]


## Load real flows

Returns `(group_id, state, label)` - `group_id` is `(directory,
scenario)`, used for a **group-level** train/test split next: flows from
the same live-test run happen in one tight time window under correlated
conditions, so splitting at the individual flow level would leak
near-duplicate examples across train/test.

In [ ]:
def load_real_flows():
    """Returns a list of (group_id, state, label) - group_id is
    (directory, scenario_name), used for a GROUP-level train/test split
    below: flows from the same live-test run happen in one tight time
    window under correlated conditions, so splitting at the individual
    flow level would leak near-duplicate examples across train/test."""
    examples = []
    for d in RESULT_DIRS:
        for path in glob.glob(os.path.join(d, "*.json")):
            if os.path.basename(path) == "summary.json":
                continue
            data = json.load(open(path))
            label = int(data["is_attack"])
            scenario = data["name"]
            for flow in data["flows"]:
                state = []
                for key in MODEL_KEYS:
                    m = flow["models"].get(key, {})
                    if m.get("available"):
                        state.append(float(m["prediction"]))
                    else:
                        state.append(0.5)  # unavailable - matches the dashboard's own degrade-gracefully convention
                examples.append(((d, scenario), np.array(state, dtype=np.float64), label))
    return examples


## Group-level, label-stratified train/test split

Stratified by label so a benign scenario-run always lands in the test
set - with few source directories there are only 1-2 benign-labeled
groups total (one per directory, vs 5 attack scenario types), so an
unstratified split can easily hold out zero of them and leave
specificity undefined.

In [ ]:
def group_train_test_split(examples, test_fraction=0.3, seed=0):
    """Stratifies the group-level split by label so a benign scenario-run
    always lands in the test set - with few source directories there are
    only 1-2 benign-labeled groups total (one per directory, vs 5 attack
    scenario types), so an unstratified split can easily hold out zero of
    them and leave specificity undefined."""
    by_label = {0: [], 1: []}
    label_of_group = {}
    for g, _, y in examples:
        label_of_group[g] = y
    for g, y in label_of_group.items():
        by_label[y].append(g)

    rng = np.random.default_rng(seed)
    test_groups = set()
    for label, groups in by_label.items():
        groups = sorted(groups)
        rng.shuffle(groups)
        n_test = max(1, round(len(groups) * test_fraction))
        test_groups.update(groups[:n_test])

    train = [(s, y) for g, s, y in examples if g not in test_groups]
    test = [(s, y) for g, s, y in examples if g in test_groups]
    return train, test, test_groups


## Sanity-check the pool before trusting it

Run this on any directory you're considering adding to `RESULT_DIRS` -
if a family model's per-scenario profile doesn't roughly match the
existing pool's, it's a different checkpoint and doesn't belong in the
same training set.

In [ ]:
import json as _json

def spot_check(directory):
    summary = _json.load(open(f"{directory}/summary.json"))
    print(f"--- {directory} ---")
    for s in summary["scenarios"]:
        ms = s["model_scores"]
        print(f"  {s['name']:<18} raw_flood={ms.get('raw_flood', 0)*100:.1f}%  "
              f"reflection={ms.get('reflection', 0)*100:.1f}%  "
              f"connection={ms.get('connection_application_layer', 0)*100:.1f}%")

for d in RESULT_DIRS:
    spot_check(d)


## Reward functions

`asymmetric_reward` is the operationally realistic one for an IDS:
missing a real attack (false negative) costs more than a false alarm
(false positive) - a cost ratio a plain accuracy-trained classifier has
no way to express.

In [ ]:
# ---------------------------------------------------------------------
# Reward functions
# ---------------------------------------------------------------------

def symmetric_reward(action, label):
    return 1.0 if action == label else -1.0


def asymmetric_reward(action, label):
    """Operationally realistic for an IDS: missing a real attack (false
    negative) costs more than a false alarm (false positive) - a cost
    ratio a plain accuracy-trained classifier has no way to express."""
    if action == label:
        return 1.0
    return -2.0 if (label == 1 and action == 0) else -1.0  # miss vs false alarm


## Tiny numpy MLP Q-network

No torch dependency - kept minimal and easy to verify.

In [ ]:
# ---------------------------------------------------------------------
# Tiny numpy MLP Q-network (same architecture as rl_retrain_policy.py)
# ---------------------------------------------------------------------

class QNetwork:
    def __init__(self, n_features, n_hidden=16, n_actions=2, lr=0.02, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.5, size=(n_features, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.normal(0, 0.5, size=(n_hidden, n_actions))
        self.b2 = np.zeros(n_actions)
        self.lr = lr

    def forward(self, x):
        z1 = x @ self.W1 + self.b1
        h = np.tanh(z1)
        q = h @ self.W2 + self.b2
        return q, h, z1

    def act(self, x, epsilon):
        if RNG.random() < epsilon:
            return RNG.integers(0, 2)
        q, _, _ = self.forward(x)
        return int(np.argmax(q))

    def train_step(self, x, action, target):
        q, h, z1 = self.forward(x)
        error = q[action] - target

        d_q = np.zeros_like(q)
        d_q[action] = error
        d_W2 = np.outer(h, d_q)
        d_b2 = d_q
        d_h = d_q @ self.W2.T
        d_z1 = d_h * (1 - np.tanh(z1) ** 2)
        d_W1 = np.outer(x, d_z1)
        d_b1 = d_z1

        self.W2 -= self.lr * d_W2
        self.b2 -= self.lr * d_b2
        self.W1 -= self.lr * d_W1
        self.b1 -= self.lr * d_b1


## Training loop - class-balanced sampling

Samples benign and attack flows with **equal** probability regardless of
the pool's natural imbalance (~80/20 attack-heavy, since 5 of 6
`simulate_attacks.py` scenario types are attacks). Without this, a
reward-maximizing policy just learns the marginal majority class and
collapses to "always predict ATTACK" - identical to the degenerate
`OR_any_flags_attack` baseline below. This was the actual failure mode
before adding the balancing - worth keeping in mind if you extend this
with your own, differently-balanced data.

In [ ]:
def train(train_examples, reward_fn, n_steps=60_000, epsilon_start=0.3, epsilon_end=0.02, seed=0):
    """Samples benign and attack flows with EQUAL probability regardless
    of the pool's natural imbalance (~80/20 attack-heavy, since 5 of 6
    simulate_attacks.py scenario types are attacks) - without this, a
    reward-maximizing policy just learns the marginal majority class and
    collapses to "always predict ATTACK", identical to the degenerate
    OR_any_flags_attack baseline. Checked directly: this was the actual
    failure mode before this fix."""
    net = QNetwork(n_features=len(MODEL_KEYS), seed=seed)
    by_label = {0: [s for s, y in train_examples if y == 0],
                1: [s for s, y in train_examples if y == 1]}
    for step in range(n_steps):
        epsilon = epsilon_start + (epsilon_end - epsilon_start) * (step / n_steps)
        label = int(RNG.random() < 0.5)
        state = by_label[label][RNG.integers(0, len(by_label[label]))]
        action = net.act(state, epsilon)
        r = reward_fn(action, label)
        net.train_step(state, action, r)
    return net


In [ ]:
def evaluate(predict_fn, examples):
    tp = fp = tn = fn = 0
    for state, label in examples:
        pred = predict_fn(state)
        if pred == 1 and label == 1:
            tp += 1
        elif pred == 1 and label == 0:
            fp += 1
        elif pred == 0 and label == 0:
            tn += 1
        else:
            fn += 1
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    accuracy = (tp + tn) / len(examples)
    return dict(accuracy=accuracy, recall=recall, specificity=specificity, tp=tp, fp=fp, tn=tn, fn=fn)


## Run: load data, split, baselines, then both RL combiners

In [ ]:
def main():
    examples = load_real_flows()
    print(f"loaded {len(examples)} real labeled flows across {len(RESULT_DIRS)} live-test directories")

    train_ex, test_ex, test_groups = group_train_test_split(examples)
    train_pairs = [(s, y) for _, s, y in train_ex] if False else train_ex
    print(f"train flows: {len(train_ex)}, test flows: {len(test_ex)} (held out {len(test_groups)} whole scenario-runs, not individual flows)")

    print("\n" + "=" * 78)
    print("baselines on the held-out test set")
    print("=" * 78)

    for i, key in enumerate(MODEL_KEYS):
        pred_fn = lambda s, i=i: int(round(s[i])) if s[i] != 0.5 else 0
        m = evaluate(pred_fn, test_ex)
        print(f"{key:<30} acc={m['accuracy']:.3f}  recall={m['recall']:.3f}  specificity={m['specificity']:.3f}")

    majority_fn = lambda s: int(sum(1 for v in s if v == 1.0) > sum(1 for v in s if v == 0.0))
    or_fn = lambda s: int(any(v == 1.0 for v in s))
    and_fn = lambda s: int(all(v != 0.0 for v in s))
    for name, fn in [("majority_vote", majority_fn), ("OR_any_flags_attack", or_fn), ("AND_all_flag_attack", and_fn)]:
        m = evaluate(fn, test_ex)
        print(f"{name:<30} acc={m['accuracy']:.3f}  recall={m['recall']:.3f}  specificity={m['specificity']:.3f}")

    print("\n" + "=" * 78)
    print("RL combiner - symmetric reward (+1 correct / -1 incorrect)")
    print("=" * 78)
    net_sym = train(train_ex, symmetric_reward)
    pred_sym = lambda s: int(np.argmax(net_sym.forward(s)[0]))
    m = evaluate(pred_sym, test_ex)
    print(f"acc={m['accuracy']:.3f}  recall={m['recall']:.3f}  specificity={m['specificity']:.3f}  (tp={m['tp']} fp={m['fp']} tn={m['tn']} fn={m['fn']})")

    print("\n" + "=" * 78)
    print("RL combiner - asymmetric reward (miss an attack costs 2x a false alarm)")
    print("=" * 78)
    net_asym = train(train_ex, asymmetric_reward)
    pred_asym = lambda s: int(np.argmax(net_asym.forward(s)[0]))
    m = evaluate(pred_asym, test_ex)
    print(f"acc={m['accuracy']:.3f}  recall={m['recall']:.3f}  specificity={m['specificity']:.3f}  (tp={m['tp']} fp={m['fp']} tn={m['tn']} fn={m['fn']})")


if __name__ == "__main__":
    main()

main()
